In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import math
import copy
from utils import *
import matplotlib.pyplot as plt

from diffusion import *
from diff_separate import *
from data_loader import *
from combined_data_loader import *
from Optimization_single_node import *
from tqdm.auto import tqdm
from scenarios_reduce import *

In [2]:
class Args:
    def __init__(self):
        self.T = 24
        self.base_mva = 100.0
        self.capacity_scale = 8
        self.ramp_rate = 0.5
        self.voll = 200.0
        self.vosp = 50.0
        self.M_beta = 1e4

        self.reserve_up_ratio = 0.05
        self.reserve_dn_ratio = 0.02
        self.rt_up_ratio = 3.0
        self.rt_dn_ratio = 0.5

        self.device = "cuda"
        self.epochs = 1
        self.dfl_train_batch_size = 16
        self.dfl_test_batch_size = 4
        self.lr = 1e-8
        self.solver = "ECOS"

        # diffusion sampling control
       # self.sample_chunk = 1              # 兼容旧逻辑
        self.train_sample_chunk = 10       # 训练时小一点，省显存
        self.test_sample_chunk = 50        # 测试时可以大一点，加速
        self.cpu_offload = False           # 训练必须 False；测试可选 True
        self.shared_noise = True

        self.mode = "ddim"                 # "ddpm" or "ddim"
        self.n_steps = 50                  # only for DDIM
        self.eta = 0.0                     # only for DDIM
        self.trunc_steps = 1               # train建议1~3；test通常0

        # scenario numbers
        self.N_scen = 20                   # OptNet 真正求解使用的场景数 K
        self.S_full = 200                  # diffusion 先生成的大候选池 S
        self.K_rand = 10                   # K里随机保留多少条
        # filter / diversity
        self.tau_gumbel = 0.1
        self.eps_uniform = 0.1
        self.lambda_div = 1e5
        self.div_type = "kl"

        # training stages
        self.filter_epochs = 5
        self.filter_lr = 1e-3
        self.dfl_epochs = 1
        self.dfl_lr = 1e-6
        self.diffusion_mode = "ddpm" 
        # optional
        self.clip_predictor = 1.0
        self.run_stage2 = False
        self.run_stage3 = False

args = Args()

In [ ]:
set_seed(42)
DTYPE = torch.float64
data_path = "../Data/load_data_city_4_2.csv"
target_nodes = [f"4-2-{i}" for i in range(11)]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data=pd.read_csv(data_path)
data["DATETIME"] = pd.to_datetime(data["DATETIME"], errors="coerce")
data_2022 = data[data["DATETIME"].dt.year == 2022].copy()
Lmin, Lmax = system_hourly_load_minmax(data_2022, datetime_col="DATETIME",node_cols=target_nodes)
Lmax_total=Lmax.sum(0)# (24,)
Lmin_total=Lmin.sum(0) # (24,)
eps_search=pd.read_csv('../Result/eps_search.csv')
eps=int(eps_search[eps_search['model']=='diffusion_s']['eps'])
args.eps_value=eps

In [7]:
set_seed(42)
models_s, handlers_s, pack_data_s = run_diffusion_single(
    data_path=data_path,
    node_cols=target_nodes,
    device=device,
    epochs=500,
    batch_size=128,
    lr=1e-3,
    patience=50,
    train_length=8760,
    val_ratio=0.2,
    seed=42,
    T=100,
    time_dim=128,
    hidden_ch=256,
    n_layers=3,
    dropout=0.0,
    find_best_ztemp=True,
    z_temp_grid=(0.5,0.75,1,1.25,1.5),
    ztemp_n_samples=50,
    ztemp_batch_size=303,
    ztemp_quantiles=[0.05 * i for i in range(1, 20)],
    diffusion_mode="ddpm",
    ddim_eta=0.0,
)

Epoch    1 | train=1.986488 | val=0.752371
Epoch   10 | train=0.458013 | val=0.518472
Epoch   20 | train=0.290747 | val=0.277155
Epoch   30 | train=0.252404 | val=0.247401
Epoch   40 | train=0.216043 | val=0.276832
Epoch   50 | train=0.263356 | val=0.313369
Epoch   60 | train=0.234155 | val=0.264213
Epoch   70 | train=0.187793 | val=0.190851
Epoch   80 | train=0.205603 | val=0.205226
Epoch   90 | train=0.178641 | val=0.216371
Epoch  100 | train=0.191645 | val=0.251227
Epoch  110 | train=0.191469 | val=0.232447
Epoch  120 | train=0.203419 | val=0.181583
Epoch  130 | train=0.160989 | val=0.249810
Epoch  140 | train=0.168243 | val=0.200198
Epoch  150 | train=0.188628 | val=0.190033
Epoch  160 | train=0.180761 | val=0.193461
Epoch  170 | train=0.167266 | val=0.238067
Epoch  180 | train=0.188610 | val=0.231588
Epoch  190 | train=0.180302 | val=0.216422
Epoch  200 | train=0.148298 | val=0.204229
Epoch  210 | train=0.162423 | val=0.189150
Epoch  220 | train=0.152109 | val=0.230198
Epoch  230 

In [10]:
set_seed(0)
window_pack_diff_train = sample_window_diffusion_single(
    models_s, handlers_s, pack_data_s, target_nodes=target_nodes,
    split="train",mode="ddpm", horizon_days=292, start_day=0, n_samples=200, seq_len=24
)

set_seed(0)
window_pack_diff_val = sample_window_diffusion_single(
    models_s, handlers_s, pack_data_s, target_nodes=target_nodes,
    split="val",mode="ddpm", horizon_days=73, start_day=0, n_samples=200, seq_len=24
)

set_seed(0)
window_pack_diff_test = sample_window_diffusion_single(
    models_s, handlers_s, pack_data_s, target_nodes=target_nodes,
    split="test",mode="ddpm", horizon_days=303, start_day=0, n_samples=200, seq_len=24
)

In [11]:
with open('../Result/Diffusion/window_pack_s_diff_train.pkl','wb') as f:
    pickle.dump(window_pack_diff_train,f)

with open('../Result/Diffusion/window_pack_s_diff_test.pkl','wb') as f:
    pickle.dump(window_pack_diff_test,f)

with open('../Result/Diffusion/window_pack_s_diff_val.pkl','wb') as f:
    pickle.dump(window_pack_diff_val,f)

In [ ]:
import pickle
with open('../Result/Diffusion/models_s.pkl', 'wb') as f:
    pickle.dump(models_s, f)
with open('../Result/Diffusion/handlers_s.pkl', 'wb') as f:
    pickle.dump(handlers_s, f)
with open('../Result/Diffusion/pack_data_s.pkl', 'wb') as f:
    pickle.dump(pack_data_s, f)